In [38]:
import librosa
import soundfile as sf
from transformers import Wav2Vec2Model, Wav2Vec2Processor
import torch
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
hf_token = os.environ["HF_TOKEN"]
login(token=hf_token)

# audio to numpy
#audio, sr = librosa.load("Stimuli\Raw_resampled\dash-tash_F0_11_VOT_11.wav", sr=16000)
#sf.write("dash-tash_11_resampled.wav", audio, 16000)

# load a soundfile in 16kHz
dash, sr = librosa.load("Stimuli\Continua\dash-tash\dash-tash_F0_07_VOT_07.wav", sr=16000)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [39]:
# preprocess numpy audio to tensors
pre_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
dash_tensor = pre_processor(dash, sampling_rate=sr, return_tensors='pt')

print(dash_tensor)
print(dash_tensor['input_values'].shape)

{'input_values': tensor([[0.0023, 0.0023, 0.0023,  ..., 0.0065, 0.0094, 0.0023]])}
torch.Size([1, 19831])


In [29]:
# run CNN and 12-layer transformer
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")

with torch.no_grad():
    embeddings = model(**dash_tensor, output_hidden_states=True)


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [40]:
from transformers import Wav2Vec2ForCTC
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

logits = model(**dash_tensor).logits
 
# take argmax and decode
predicted_ids = torch.argmax(logits, dim=-1)
transcription = pre_processor.batch_decode(predicted_ids)

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [41]:
print(transcription)

['TASHE']


In [ ]:
transcripts = []
steps = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11"]
continua = 

path = os.path.join("Stimuli", "Continua", "task-dask", "task-dask_F0_01_VOT_01.wav")

for step in steps:
    continuum_step, sr = librosa.load(f"Stimuli/Continua/task-dask/task-dask_F0_{step}_VOT_{step}.wav", sr=16000)
    
    tensor = pre_processor(continuum_step, sampling_rate=sr, return_tensors='pt')
    logits = model(**tensor).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = pre_processor.batch_decode(predicted_ids)
    transcripts.append(transcription)

print(transcripts)

In [4]:
# output of the first hidden layer (CNN)
print(embeddings.hidden_states[0])

print(embeddings.hidden_states[0].shape)

tensor([[[-0.1989, -0.1269, -0.3311,  ...,  0.1660,  0.5340,  0.1340],
         [ 0.0525,  0.3866, -0.2554,  ...,  0.8195,  0.4377, -0.2764],
         [ 0.5461,  0.1322, -0.0841,  ..., -0.2403, -0.2208, -0.8773],
         ...,
         [ 0.4066,  0.0362,  0.0397,  ..., -0.0381,  0.3028,  0.1629],
         [ 0.4060,  0.0885,  0.0225,  ...,  0.0708,  0.5041,  0.2288],
         [ 0.5137, -0.0737,  0.2317,  ...,  0.0112,  0.5276,  0.2689]]])
torch.Size([1, 60, 768])


In [5]:
start_frame = int(0.05 // 0.02)
end_frame = int(0.056 // 0.02)

# [0] -> layer
print(embeddings.hidden_states[0][0, start_frame:end_frame +1, :].mean(dim=0))

# should be 13 (CNN + 12 transformer layers)
#print(len(embeddings.hidden_states))

# CNN output shape
#print(embeddings.hidden_states[0].shape)

tensor([ 5.4615e-01,  1.3218e-01, -8.4057e-02,  1.7350e-01,  6.5498e-02,
        -1.6236e-01, -5.8455e-01, -3.1418e-01,  5.8121e-02,  2.2033e-01,
        -2.7124e-01, -1.8707e-01, -3.6759e-01, -1.1675e-01, -4.8621e-02,
        -6.4664e-01,  2.7818e-01, -5.5293e-01,  9.6941e-02,  5.6413e-01,
        -2.5777e-01, -4.1815e-02,  8.3692e-01, -6.1369e-01, -2.1029e-01,
         4.2523e-01, -4.6270e-01,  2.5625e-01, -1.6674e-01,  5.4073e-01,
        -7.0113e-02,  5.6266e-01,  4.4827e-01,  9.9447e-02, -8.7397e-02,
         2.0022e-01, -6.4087e-02,  1.4345e-02,  3.9004e-01, -1.4718e-01,
        -4.0586e-01, -4.0588e-01,  4.7128e-01,  2.1687e-02, -8.1906e-02,
         2.8188e-02,  1.8218e-01,  1.0382e-01,  3.3525e-01,  1.9694e-01,
         9.9041e-02,  4.4532e-01, -9.8609e-02, -1.6524e-01, -3.6872e-01,
        -3.5161e-02, -2.7208e-02,  3.3514e-01,  6.5981e-02, -3.0933e-01,
        -5.3530e-01,  2.1168e-01, -1.9621e-01, -4.5457e-02,  6.1101e-02,
         1.7361e-01, -6.5529e-02, -1.7308e-01,  6.0

NameError: name 'p' is not defined